# 01 — Archive GTFS-Realtime Feeds
Notebook version of the archiver — useful for testing and one-off runs.
For continuous archiving, run `python scripts/archive_gtfsrt.py` instead.

In [ ]:
# Cell 1 — Install dependencies
!pip install requests gtfs-realtime-bindings protobuf pyyaml pandas

In [ ]:
# Cell 2 — Load config and print feed URLs
import yaml
from pathlib import Path

REPO_ROOT = Path.cwd().parent
with open(REPO_ROOT / 'config' / 'feeds.yaml') as f:
    config = yaml.safe_load(f)

feeds = config['gtfs_realtime']
print('Feed URLs:')
for name, url in feeds.items():
    print(f'  {name}: {url}')
print(f"\nFetch interval: {config['fetch_interval_seconds']}s")

In [ ]:
# Cell 3 — Fetch trip_updates → parse protobuf → DataFrame
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2

def fetch_feed(url):
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    fm = gtfs_realtime_pb2.FeedMessage()
    fm.ParseFromString(resp.content)
    return fm

fm_tu = fetch_feed(feeds['trip_updates'])
ts = int(datetime.now().timestamp())

rows = []
for entity in fm_tu.entity:
    if not entity.HasField('trip_update'):
        continue
    tu = entity.trip_update
    for stu in tu.stop_time_update:
        rows.append({
            'trip_id': tu.trip.trip_id,
            'route_id': tu.trip.route_id,
            'stop_id': stu.stop_id,
            'delay_seconds': stu.arrival.delay if stu.HasField('arrival') else None,
            'timestamp': ts,
        })

df_tu = pd.DataFrame(rows)
print(f'trip_updates: {len(df_tu)} records')
df_tu.head()

In [ ]:
# Cell 4 — Fetch vehicle_positions → parse → DataFrame
fm_vp = fetch_feed(feeds['vehicle_positions'])

rows = []
for entity in fm_vp.entity:
    if not entity.HasField('vehicle'):
        continue
    vp = entity.vehicle
    rows.append({
        'vehicle_id': vp.vehicle.id,
        'trip_id': vp.trip.trip_id,
        'lat': vp.position.latitude,
        'lon': vp.position.longitude,
        'bearing': vp.position.bearing if vp.position.HasField('bearing') else None,
        'timestamp': ts,
    })

df_vp = pd.DataFrame(rows)
print(f'vehicle_positions: {len(df_vp)} records')
df_vp.head()

In [ ]:
# Cell 5 — Fetch service_alerts → parse → DataFrame
fm_sa = fetch_feed(feeds['service_alerts'])

rows = []
for entity in fm_sa.entity:
    if not entity.HasField('alert'):
        continue
    alert = entity.alert
    cause = gtfs_realtime_pb2.Alert.Cause.Name(alert.cause)
    effect = gtfs_realtime_pb2.Alert.Effect.Name(alert.effect)
    header = alert.header_text.translation[0].text if alert.header_text.translation else ''
    rows.append({
        'alert_id': entity.id,
        'cause': cause,
        'effect': effect,
        'header': header,
        'timestamp': ts,
    })

df_sa = pd.DataFrame(rows)
print(f'service_alerts: {len(df_sa)} records')
df_sa.head()

In [ ]:
# Cell 6 — Save all three feeds to source_files/gtfs_realtime/ with today's date subfolder
import json
from datetime import datetime

now = datetime.now()
date_str = now.strftime('%Y-%m-%d')
time_str = now.strftime('%H-%M-%S')

GTFSRT_DIR = REPO_ROOT / 'source_files' / 'gtfs_realtime'

saved_files = []
for feed_name, df in [('trip_updates', df_tu), ('vehicle_positions', df_vp), ('service_alerts', df_sa)]:
    out_dir = GTFSRT_DIR / feed_name / date_str
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{time_str}.json'
    records = df.to_dict(orient='records')
    with open(out_path, 'w') as f:
        json.dump(records, f, separators=(',', ':'))
    saved_files.append((feed_name, str(out_path.relative_to(REPO_ROOT)), len(records)))
    print(f'Saved {feed_name}: {len(records)} records → {out_path.relative_to(REPO_ROOT)}')

In [ ]:
# Cell 7 — Summary
from datetime import datetime

print('=' * 60)
print(f'Archive run complete: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 60)
for feed_name, path, count in saved_files:
    print(f'  {feed_name:<20} {count:>5} records  →  {path}')
print()
print(f'Total records archived: {sum(c for _, _, c in saved_files)}')
print()
print('Next step: run notebooks/02_load_static_gtfs.ipynb')